# 04 Semantic change detection on SECOND

SECOND has 512x512 image pairs of the same place at two dates. Land-cover classes are labelled only where
something changed (unchanged pixels are white), so the model has to look at both images together.

Model: a Siamese U-Net.
- One shared ResNet34 encoder (started from the UAVid model) reads both images.
- The U-Net decoder labels the land cover of each image.
- A small change head compares the two images' features at every scale (absolute difference) and predicts changed / unchanged.
- Final output: where the change head says "changed", the land-cover class before and after.

Runs on **Colab** (files in Google Drive) or **Kaggle** (files in the notebook output).
On Kaggle, attach `unet_r34_best.pth` as a dataset (and optionally `second_siamese_last.pth` to resume),
turn on GPU T4 and Internet, then use Save Version > Save & Run All so it trains in the background.

In [ ]:
import os, glob, shutil

ON_KAGGLE = os.path.exists('/kaggle/input')
if ON_KAGGLE:
    DRIVE = '/kaggle/working/drone-mapping'          # saved with the notebook output
    SECOND_DRIVE = '/tmp/second_download'            # big files stay out of the output
    os.makedirs(f'{DRIVE}/models', exist_ok=True)
    # Copy weights and checkpoints attached as Kaggle datasets
    for f in glob.glob('/kaggle/input/**/*.pth', recursive=True):
        dst = f'{DRIVE}/models/{os.path.basename(f)}'
        if not os.path.exists(dst):
            shutil.copy(f, dst)
            print('Copied', os.path.basename(f))
    # A SECOND zip attached as a dataset is used instead of downloading
    os.makedirs(SECOND_DRIVE, exist_ok=True)
    for f in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        dst = f'{SECOND_DRIVE}/{os.path.basename(f)}'
        if not os.path.exists(dst):
            os.symlink(f, dst)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/drone-mapping'
    SECOND_DRIVE = f'{DRIVE}/data/second'

OUT = f'{DRIVE}/outputs/second'
for d in (SECOND_DRIVE, f'{DRIVE}/models', OUT):
    os.makedirs(d, exist_ok=True)
WORKERS = min(4, os.cpu_count() or 2)
print('Platform:', 'Kaggle' if ON_KAGGLE else 'Colab', '| data loader workers:', WORKERS)

!pip install -q segmentation-models-pytorch gdown
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Download and inspect

In [ ]:
import glob, zipfile

# Google Drive file ids from the SECOND page (https://captain-whu.github.io/SCD)
FILE_IDS = ['1mN8jzCKKK27p3ODGoDgepjiRYGQpB34u', '1QlAdzrHpfBIOZ6SK78yHF2i1u6tikmBc']
ZIP = f'{SECOND_DRIVE}/second.zip'

if not glob.glob(f'{SECOND_DRIVE}/*.zip') and not glob.glob(f'{SECOND_DRIVE}/*.rar') and not glob.glob(f'{SECOND_DRIVE}/*.7z'):
    for fid in FILE_IDS:
        !gdown {fid} -O "{ZIP}"
        if os.path.exists(ZIP) and zipfile.is_zipfile(ZIP):
            break
        print('That link did not give a zip, trying the next one')
archives = sorted(glob.glob(f'{SECOND_DRIVE}/*.zip') + glob.glob(f'{SECOND_DRIVE}/*.rar') + glob.glob(f'{SECOND_DRIVE}/*.7z'))
print('Archives:', [(os.path.basename(a), f'{os.path.getsize(a) / 1e9:.2f} GB') for a in archives])
assert archives, ('Download failed (Google Drive limits busy files). Open the link from the SECOND page in your browser, '
                  f'download the file, and upload it to {SECOND_DRIVE}, then rerun this cell.')

In [ ]:
import shutil, subprocess

ROOT = '/tmp/second' if ON_KAGGLE else '/content/second'


def run(cmd):
    try:
        return subprocess.run(cmd, stdout=subprocess.DEVNULL).returncode == 0
    except FileNotFoundError:            # tool not installed
        return False


def apt(pkg):
    subprocess.run(f'apt-get -qq install -y {pkg} > /dev/null 2>&1 || '
                   f'(apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y {pkg} > /dev/null 2>&1)', shell=True)


def extract_rar(archive, dest):
    # 7z on Colab/Kaggle cannot read the compression used in SECOND's .rar ("Unsupported Method"),
    # so use the official unrar, then bsdtar, then rarlab's own build
    if shutil.which('unrar') is None:
        apt('unrar')
    if run(['unrar', 'x', '-o+', '-inul', archive, dest + '/']):
        return True
    print('  unrar not available or failed, trying bsdtar')
    if shutil.which('bsdtar') is None:
        apt('libarchive-tools')
    if run(['bsdtar', '-xf', archive, '-C', dest]):
        return True
    print('  bsdtar failed, downloading unrar from rarlab.com')
    for v in ('712', '711', '710', '701', '700'):
        if run(['wget', '-q', f'https://www.rarlab.com/rar/rarlinux-x64-{v}.tar.gz', '-O', '/tmp/rar.tgz']):
            run(['tar', '-xzf', '/tmp/rar.tgz', '-C', '/tmp'])
            if run(['/tmp/rar/unrar', 'x', '-o+', '-inul', archive, dest + '/']):
                return True
    return False


def extract(archive, dest):
    """Unpack a .zip, .rar or .7z into dest. Returns True on success."""
    print('Extracting', os.path.basename(archive), '...')
    os.makedirs(dest, exist_ok=True)
    low = archive.lower()
    if low.endswith('.zip'):
        return run(['unzip', '-q', '-o', archive, '-d', dest])
    if low.endswith('.rar'):
        return extract_rar(archive, dest)
    if shutil.which('7z') is None:
        apt('p7zip-full')
    return run(['7z', 'x', '-y', f'-o{dest}', archive])


DONE = f'{ROOT}/.extracted'
if not os.path.exists(DONE):
    shutil.rmtree(ROOT, ignore_errors=True)        # clear any half-finished earlier attempt
    os.makedirs(ROOT)
    for a in archives:
        assert extract(a, ROOT), f'Could not extract {a}'
    # The SECOND zip contains more archives (a .rar for train, a .zip for test). Each one gets its own
    # folder, because the .rar has im1/im2/label1/label2 at the top with no parent folder.
    for _ in range(3):
        inner = [p for p in glob.glob(f'{ROOT}/**/*', recursive=True)
                 if p.lower().endswith(('.zip', '.rar', '.7z')) and '__MACOSX' not in p and os.path.isfile(p)]
        if not inner:
            break
        for a in inner:
            dest = os.path.splitext(a)[0]
            assert extract(a, dest), f'Could not extract {a}. Paste the output above to Claude.'
            os.remove(a)              # the original is still in the download folder; this frees disk space
    shutil.rmtree(f'{ROOT}/__MACOSX', ignore_errors=True)
    open(DONE, 'w').close()
!du -sh {ROOT}
print('Data in', ROOT)

In [ ]:
import re
import cv2
import numpy as np

IMG_EXT = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
dirs = {}
for d, _, files in os.walk(ROOT):
    if '__MACOSX' in d:
        continue
    f = sorted(x for x in files if x.lower().endswith(IMG_EXT) and not x.startswith('._'))
    if f:
        dirs[d] = f
for d, f in sorted(dirs.items()):
    print(f'  {os.path.relpath(d, ROOT):50s} {len(f):6d} files   e.g. {f[0]}')

# SECOND folders are called im1, im2, label1, label2 (sometimes label1_rgb / label2_rgb)
ROLE_NAMES = {'im1': 'img1', 'im2': 'img2', 'label1': 'lab1', 'label2': 'lab2',
              'label1_rgb': 'lab1', 'label2_rgb': 'lab2'}
groups = {}
for d in dirs:
    role = ROLE_NAMES.get(os.path.basename(d).lower())
    if role:
        groups.setdefault(os.path.dirname(d), {}).setdefault(role, d)   # first match wins

def by_stem(folder):
    return {os.path.splitext(f)[0]: os.path.join(folder, f) for f in dirs[folder]}

PAIRS = []
for parent, r in sorted(groups.items()):
    if {'img1', 'img2', 'lab1', 'lab2'} <= set(r):
        counts = {k: len(dirs[r[k]]) for k in ('img1', 'img2', 'lab1', 'lab2')}
        if len(set(counts.values())) > 1:
            print(f'WARNING {os.path.relpath(parent, ROOT)}: folders have different file counts {counts}')
        empty = [os.path.join(r[k], f) for k in ('img1', 'img2', 'lab1', 'lab2') for f in dirs[r[k]]
                 if os.path.getsize(os.path.join(r[k], f)) == 0]
        if empty:
            print(f'WARNING {os.path.relpath(parent, ROOT)}: {len(empty)} empty files, e.g. {empty[0]}. '
                  'Extraction was incomplete.')
        m = [by_stem(r[k]) for k in ('img1', 'img2', 'lab1', 'lab2')]
        stems = sorted(set(m[0]) & set(m[1]) & set(m[2]) & set(m[3]))
        PAIRS += [{'stem': f'{os.path.relpath(parent, ROOT)}/{s}', 'img1': m[0][s], 'img2': m[1][s],
                   'lab1': m[2][s], 'lab2': m[3][s]} for s in stems]
        print(f'{os.path.relpath(parent, ROOT)}: {len(stems)} labelled pairs')
assert PAIRS, 'No im1/im2/label1/label2 folders found. Check the folder list above.'
print('Total labelled pairs:', len(PAIRS))

In [ ]:
# Label colours used by SECOND (from the authors' code, github.com/ggsDing/Bi-SRNet)
CLASSES = ['Water', 'Ground', 'LowVeg', 'Tree', 'Building', 'Playground']
NUM_CLASSES = len(CLASSES)
SECOND_RGB = [(255, 255, 255), (0, 0, 255), (128, 128, 128), (0, 128, 0), (0, 255, 0), (128, 0, 0), (255, 0, 0)]
PALETTE = np.array(SECOND_RGB, np.uint8)        # index 0 = unchanged (white), 1..6 = classes
IGNORE = 255


def read_image(path):
    return cv2.cvtColor(cv2.imread(path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)


def read_label(path):
    """Returns 0 = unchanged, 1..6 = land-cover class, 255 = unknown colour."""
    m = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if m is None:
        raise FileNotFoundError(path)
    if m.ndim == 3:
        rgb = cv2.cvtColor(m[..., :3], cv2.COLOR_BGR2RGB)
        out = np.full(rgb.shape[:2], IGNORE, np.uint8)
        for i, c in enumerate(SECOND_RGB):
            out[np.all(rgb == c, axis=-1)] = i
        return out
    out = m.astype(np.uint8).copy()
    out[out > NUM_CLASSES] = IGNORE
    return out


ex = PAIRS[0]
l1, l2 = read_label(ex['lab1']), read_label(ex['lab2'])
print('Image', read_image(ex['img1']).shape, '| label', l1.shape)
print('Values in label1:', np.unique(l1).tolist(), '| label2:', np.unique(l2).tolist())
print('Change masks agree between label1 and label2:', bool(((l1 == 0) == (l2 == 0)).all()))

# Class balance over a sample
cnt = np.zeros(NUM_CLASSES + 1, np.int64)
for p in PAIRS[::max(1, len(PAIRS) // 200)]:
    for k in ('lab1', 'lab2'):
        cnt += np.bincount(read_label(p[k]).ravel(), minlength=256)[:NUM_CLASSES + 1]
share = cnt / cnt.sum()
print(f'\nUnchanged {100 * share[0]:.1f}% of pixels. Changed pixels by class:')
for c, s in zip(CLASSES, share[1:]):
    print(f'  {c:10s} {100 * s:5.1f}%')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(3, 4, figsize=(16, 12))
for r, p in enumerate(PAIRS[:3]):
    for a, im, t in zip(ax[r], [read_image(p['img1']), read_image(p['img2']), PALETTE[read_label(p['lab1']) % 7],
                                PALETTE[read_label(p['lab2']) % 7]], ['before', 'after', 'label before', 'label after']):
        a.imshow(im); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()

## 2. Split

If the download includes the official test set with labels, it is used only once at the end, and 10% of the
training set is held out for validation (picking the best epoch). Then scores are comparable with papers.
If there is no labelled test set, the training pairs are split 80 / 10 / 10 instead, and scores are not
directly comparable with papers.

In [ ]:
import random

is_test = lambda p: 'test' in p['stem'].split('/')[0].lower()
OFFICIAL_TEST = [p for p in PAIRS if is_test(p)]
pool = sorted((p for p in PAIRS if not is_test(p)), key=lambda p: p['stem'])
random.Random(0).shuffle(pool)
n = len(pool)
if OFFICIAL_TEST:
    TEST, VAL, TRAIN = OFFICIAL_TEST, pool[:n // 10], pool[n // 10:]
    TEST_NAME = 'official test set'
else:
    TEST, VAL, TRAIN = pool[:n // 10], pool[n // 10: n // 5], pool[n // 5:]
    TEST_NAME = 'own 10% split'
print(f'Train {len(TRAIN)} | val {len(VAL)} | test {len(TEST)} ({TEST_NAME})')

## 3. Model and training

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)


def to_tensor(img):
    return torch.from_numpy(np.ascontiguousarray(((img.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)))


class PairDataset(Dataset):
    def __init__(self, pairs, train):
        self.pairs, self.train = pairs, train

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        p = self.pairs[i]
        a, b = read_image(p['img1']), read_image(p['img2'])
        la, lb = read_label(p['lab1']), read_label(p['lab2'])
        if self.train:
            # Same flips and rotation for both dates, so pixels stay aligned
            if random.random() < 0.5:
                a, b, la, lb = a[:, ::-1], b[:, ::-1], la[:, ::-1], lb[:, ::-1]
            if random.random() < 0.5:
                a, b, la, lb = a[::-1], b[::-1], la[::-1], lb[::-1]
            k = random.randint(0, 3)
            a, b, la, lb = (np.rot90(x, k) for x in (a, b, la, lb))
            # Different brightness per date, like real photos taken in different light
            a, b = [np.clip(x.astype(np.float32) * random.uniform(0.8, 1.2) + random.uniform(-20, 20), 0, 255)
                    if random.random() < 0.5 else x for x in (a, b)]
        change = np.where((la == IGNORE) | (lb == IGNORE), IGNORE, (la > 0).astype(np.uint8))
        # Land-cover targets: classes 1..6 become 0..5, unchanged pixels are ignored
        sa = np.where((la >= 1) & (la <= NUM_CLASSES), la.astype(np.int64) - 1, IGNORE)
        sb = np.where((lb >= 1) & (lb <= NUM_CLASSES), lb.astype(np.int64) - 1, IGNORE)
        return (to_tensor(a), to_tensor(b), torch.from_numpy(np.ascontiguousarray(change)).long(),
                torch.from_numpy(np.ascontiguousarray(sa)).long(), torch.from_numpy(np.ascontiguousarray(sb)).long())


class SiameseUNet(nn.Module):
    def __init__(self, n_classes, mid=64):
        super().__init__()
        self.seg = smp.Unet('resnet34', encoder_weights='imagenet', classes=n_classes)
        ch = self.seg.encoder.out_channels[1:]              # skip the raw image level
        self.lateral = nn.ModuleList([nn.Sequential(nn.Conv2d(c, mid, 1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
                                      for c in ch])
        self.change_head = nn.Sequential(nn.Conv2d(mid, mid, 3, padding=1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
                                         nn.Conv2d(mid, 1, 1))

    def decode(self, feats):
        try:
            d = self.seg.decoder(*feats)     # older segmentation-models-pytorch
        except TypeError:
            d = self.seg.decoder(feats)      # newer versions take a list
        return self.seg.segmentation_head(d)

    def forward(self, a, b):
        fa, fb = self.seg.encoder(a), self.seg.encoder(b)
        size = fa[1].shape[-2:]
        # Difference of the two dates' features at every scale, merged at half resolution
        diff = sum(F.interpolate(lat(torch.abs(xa - xb)), size=size, mode='bilinear', align_corners=False)
                   for lat, xa, xb in zip(self.lateral, fa[1:], fb[1:]))
        change = F.interpolate(self.change_head(diff), size=a.shape[-2:], mode='bilinear', align_corners=False)
        return change[:, 0], self.decode(fa), self.decode(fb)


device = 'cuda'
model = SiameseUNet(NUM_CLASSES)
UAVID = f'{DRIVE}/models/unet_r34_best.pth'
if os.path.exists(UAVID):
    old = torch.load(UAVID, map_location='cpu', weights_only=True)
    own = model.seg.state_dict()
    keep = {k: v for k, v in old.items() if k in own and v.shape == own[k].shape}
    model.seg.load_state_dict(keep, strict=False)
    print(f'Started from the UAVid model: {len(keep)} of {len(own)} tensors reused')
else:
    print('UAVid weights not found, starting from ImageNet encoder only')
model = model.to(device)

# Quick shape check
with torch.no_grad():
    c, sa, sb = model(torch.zeros(1, 3, 512, 512, device=device), torch.zeros(1, 3, 512, 512, device=device))
print('Output shapes:', tuple(c.shape), tuple(sa.shape), tuple(sb.shape))

In [ ]:
BATCH = 8
EPOCHS = 30

train_dl = DataLoader(PairDataset(TRAIN, True), batch_size=BATCH, shuffle=True, num_workers=WORKERS,
                      pin_memory=True, drop_last=True)
val_dl = DataLoader(PairDataset(VAL, False), batch_size=BATCH, num_workers=WORKERS)
test_dl = DataLoader(PairDataset(TEST, False), batch_size=BATCH, num_workers=WORKERS)

seg_ce = nn.CrossEntropyLoss(ignore_index=IGNORE)
chg_dice = smp.losses.DiceLoss(mode='binary', ignore_index=IGNORE)


def chg_bce(logits, target):
    m = target != IGNORE
    return F.binary_cross_entropy_with_logits(logits[m], target[m].float())


def loss_fn(c, sa, sb, change, ta, tb):
    c, sa, sb = c.float(), sa.float(), sb.float()
    seg = 0.5 * (seg_ce(sa, ta) + seg_ce(sb, tb))
    chg = chg_bce(c, change) + chg_dice(c.unsqueeze(1), change.unsqueeze(1))
    return seg + chg, seg.item(), chg.item()


opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda')

In [ ]:
def cal_kappa(hist):
    if hist.sum() == 0:
        return 0.0
    po = np.diag(hist).sum() / hist.sum()
    pe = (hist.sum(1) * hist.sum(0)).sum() / hist.sum() ** 2
    return 0.0 if pe == 1 else (po - pe) / (1 - pe)


def scd_scores(hist):
    """SECOND metrics, same maths as the authors' SCDD_eval_all. hist[pred, label], index 0 = unchanged."""
    c2 = np.array([[hist[0, 0], hist[0, 1:].sum()], [hist[1:, 0].sum(), hist[1:, 1:].sum()]], float)
    iu = np.diag(c2) / np.maximum(c2.sum(1) + c2.sum(0) - np.diag(c2), 1)
    h0 = hist.astype(float).copy(); h0[0, 0] = 0
    sek = cal_kappa(h0) * np.exp(iu[1]) / np.e
    pred_ch, lab_ch = hist[1:, :].sum(), hist[:, 1:].sum()
    sc_tp = np.diag(hist[1:, 1:]).sum()
    p, r = sc_tp / max(pred_ch, 1), sc_tp / max(lab_ch, 1)
    # Plain changed / unchanged scores
    tp, fp, fn = c2[1, 1], c2[1, 0], c2[0, 1]
    bp, br = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {'OA': float(np.diag(hist).sum() / max(hist.sum(), 1)), 'mIoU': float(iu.mean()), 'SeK': float(sek),
            'Fscd': float(2 * p * r / max(p + r, 1e-9)),
            'change_precision': float(bp), 'change_recall': float(br),
            'change_F1': float(2 * bp * br / max(bp + br, 1e-9)), 'change_IoU': float(iu[1])}


def add_to_hist(hist, pred, change, target):
    """pred: 0 = unchanged, 1..6 = class. change: 0/1/255. target: class 0..5 or 255."""
    K = hist.shape[0]
    lab = np.where(change == 1, target.astype(np.int64) + 1, 0)            # back to 0..6
    lab = np.where((change == IGNORE) | ((change == 1) & (target == IGNORE)), -1, lab)
    m = lab >= 0
    hist += np.bincount(pred[m].astype(np.int64) * K + lab[m], minlength=K * K).reshape(K, K)


def scd_maps(c, sa, sb, thr=0.5):
    """Combine the heads: 0 where unchanged, else land-cover class + 1."""
    changed = torch.sigmoid(c.float()) > thr
    pa = (sa.argmax(1) + 1) * changed
    pb = (sb.argmax(1) + 1) * changed
    return pa.cpu().numpy().astype(np.int64), pb.cpu().numpy().astype(np.int64)


@torch.no_grad()
def evaluate(dl):
    model.eval()
    K = NUM_CLASSES + 1
    hist = np.zeros((K, K), np.int64)
    for a, b, change, ta, tb in dl:
        a, b = a.to(device), b.to(device)
        with torch.amp.autocast('cuda'):
            c, sa, sb = model(a, b)
        pa, pb = scd_maps(c, sa, sb)
        for pred, t in ((pa, ta), (pb, tb)):
            add_to_hist(hist, pred, change.numpy(), t.numpy())
    return scd_scores(hist), hist

In [ ]:
import time

CKPT_LAST = f'{DRIVE}/models/second_siamese_last.pth'
CKPT_BEST = f'{DRIVE}/models/second_siamese_best.pth'
start_epoch, best_sek, history = 0, -1.0, []

if os.path.exists(CKPT_LAST):
    ck = torch.load(CKPT_LAST, map_location=device, weights_only=False)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
    start_epoch, best_sek, history = ck['epoch'] + 1, ck['best_sek'], ck['history']
    print(f'Resuming from epoch {start_epoch + 1}')

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    model.train(); seg_sum = chg_sum = 0.0
    for a, b, change, ta, tb in train_dl:
        a, b, change, ta, tb = (x.to(device, non_blocking=True) for x in (a, b, change, ta, tb))
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            c, sa, sb = model(a, b)
        loss, ls, lc = loss_fn(c, sa, sb, change, ta, tb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        seg_sum += ls; chg_sum += lc
    sched.step()

    val, _ = evaluate(val_dl)
    history.append({'epoch': epoch + 1, 'seg_loss': seg_sum / len(train_dl), 'chg_loss': chg_sum / len(train_dl), **val})
    if val['SeK'] > best_sek:
        best_sek = val['SeK']
        torch.save(model.state_dict(), CKPT_BEST)
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'scaler': scaler.state_dict(), 'epoch': epoch, 'best_sek': best_sek, 'history': history}, CKPT_LAST)
    dt = time.time() - t0
    print(f'Epoch {epoch + 1}/{EPOCHS} | seg {history[-1]["seg_loss"]:.3f} chg {history[-1]["chg_loss"]:.3f} | '
          f'val SeK {val["SeK"]:.3f} Fscd {val["Fscd"]:.3f} mIoU {val["mIoU"]:.3f} | {dt / 60:.1f} min '
          f'(about {dt * (EPOCHS - epoch - 1) / 60:.0f} min left)')

## 4. Results

In [ ]:
import json

ep = [h['epoch'] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(ep, [h['seg_loss'] for h in history], label='land cover')
ax[0].plot(ep, [h['chg_loss'] for h in history], label='change'); ax[0].set_title('Train loss'); ax[0].legend()
for k in ('SeK', 'Fscd', 'mIoU'):
    ax[1].plot(ep, [h[k] for h in history], label=k)
ax[1].set_title('Validation'); ax[1].legend()
plt.savefig(f'{OUT}/training_curves.png', dpi=120, bbox_inches='tight'); plt.show()

model.load_state_dict(torch.load(CKPT_BEST, map_location=device, weights_only=True))
best = max(history, key=lambda h: h['SeK'])
test, test_hist = evaluate(test_dl)

print(f'Best epoch {best["epoch"]} (chosen on val) | test = {TEST_NAME}, {len(TEST)} pairs\n')
print(f'{"Metric":18s} {"Val":>6s} {"Test":>6s}')
for k in ('SeK', 'Fscd', 'mIoU', 'OA', 'change_precision', 'change_recall', 'change_F1', 'change_IoU'):
    print(f'{k:18s} {best[k]:6.3f} {test[k]:6.3f}')

# Which class changes the model gets right, on test
names = ['Unchanged'] + CLASSES
recall = np.diag(test_hist) / np.maximum(test_hist.sum(0), 1)
print('\nTest recall per label class (share of each true class predicted correctly):')
for nme, r, n in zip(names, recall, test_hist.sum(0)):
    print(f'  {nme:10s} {r:.3f}   ({n:,} px)')

with open(f'{OUT}/second_results.json', 'w') as f:
    json.dump({'split': {'train': len(TRAIN), 'val': len(VAL), 'test': len(TEST)}, 'test_set': TEST_NAME,
               'best_epoch': best['epoch'],
               'val': {k: best[k] for k in test}, 'test': test, 'test_hist': test_hist.tolist(), 'classes': names,
               'history': history}, f, indent=1)

In [ ]:
@torch.no_grad()
def predict_pair(p):
    a, b, change, ta, tb = PairDataset([p], False)[0]
    with torch.amp.autocast('cuda'):
        c, sa, sb = model(a[None].to(device), b[None].to(device))
    pa, pb = scd_maps(c, sa, sb)
    return pa[0], pb[0]


# Show test pairs with a decent amount of change
shown = [p for p in TEST if (read_label(p['lab1']) == 0).mean() < 0.85][:4] or TEST[:4]
fig, ax = plt.subplots(len(shown), 6, figsize=(24, 4 * len(shown)), squeeze=False)
for r, p in enumerate(shown):
    pa, pb = predict_pair(p)
    row = [read_image(p['img1']), read_image(p['img2']), PALETTE[read_label(p['lab1']) % 7],
           PALETTE[read_label(p['lab2']) % 7], PALETTE[pa], PALETTE[pb]]
    for a, im, t in zip(ax[r], row, ['before', 'after', 'true before', 'true after', 'predicted before', 'predicted after']):
        a.imshow(im); a.set_title(t); a.axis('off')
plt.tight_layout()
plt.savefig(f'{OUT}/second_examples.jpg', dpi=70, bbox_inches='tight'); plt.show()
print('Saved to', OUT)